# Exploritary Data Analysis



# Step 0: Imports and Reading Data

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns
plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [48]:
drive.flush_and_unmount()

In [50]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/sample_stratified_250k_ready.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/sample_stratified_250k_ready.csv'

# Observations
- The essential libraries for data analysis were imported:

  - pandas and numpy for data processing.

  - matplotlib and seaborn for graphs.
- ggplot style was enabled to give the graphs a professional look.

- Display up to 200 columns for easy navigation of large tables.

- The file:
sample_stratified_250k_ready.csv
(Stratified Sample) was read from the original data.

# Step 1: Data Understanding
- Dataframe `shape`
- `head` and `tail`
- `dtypes`
- `describe`

In [ ]:
df.shape

# Observations

- Sample size (250,001 rows) is sufficient for reliable statistical analysis.

- The number of columns (73) indicates that the data is rich in features, often including flow features.

In [ ]:
df.head()

# Observations

- The first columns represent the basic flow metadata attributes:

         Src IP, Dst IP, Src Port, Dst Port, Protocol, Timestamp.

- There are several columns related to both forward and backward data flow (Fwd/Bwd), such as:

      Total Fwd Packets, Total Bwd Packets, Fwd Packet Length Mean, Bwd Packet Length Std...

- The float values ​​indicate calculated flow statistics, such as Flow Packets/s and Flow IAT Mean.

- Some values ​​(such as Packet Length) have a value of 0.0, which may indicate:

           Either very short flows (only a few packets),

         Or some attributes are unregistered or empty and need to be cleaned up later.

- The timestamp format appears to be fixed (in DD/MM/YYYY format and 12-hour clock), which will aid in later time analysis.

In [ ]:
df.columns

# Observations

- The columns are grouped into clear groups:

- Network Metadata: (Src IP, Dst IP, Protocol, Timestamp, etc.)

- Forward/Backward Flow Statistics: (Total Fwd Packets, Total Bwd Packets, Fwd Packet Length Mean, etc.)

- Flag Counts: (SYN Flag Count, PSH Flag Count, ACK Flag Count, etc.)

 - Bulk and Subflow Stats: (Fwd Bytes/Bulk Avg, Subflow Fwd Packets, etc.)

- Activity & Idle Features: (Active Mean, Idle Std, Idle Max, etc.)

-  The presence of the label in the last column indicates that the file is ready for classification, and labels have already been added (Benign / Attack).

- The presence of a large number of statistical features (such as Mean, Max, Min, Std) indicates that the data is derived from network flow analysis at the flow level, not the packet level.

- Some columns have duplicate or very similar names (e.g., Bwd Packet Length Max is duplicated twice) — the actual duplicate should be checked later.

In [ ]:
df.dtypes

***Data Types for each column in the DataFrame.***
# Observations

- There are 73 columns, as in the previous results.

- The data types are distributed as follows:

- object: Text columns such as Src IP, Dst IP, Timestamp, and label.

 - int64: Integer columns (such as Src Port, Dst Port, Protocol, Flag Counts, etc.).

- float64: Continuous statistical columns such as Flow Duration, Idle Mean, Packet Length Mean, etc.

 - The label column is of the object type, meaning that labels are written as text, such as "Benign," "DDoS," "Mirai," etc.

- This is convenient for the EDA phase and will later be converted to numbers during training (Label Encoding).

In [ ]:
df.describe()

 **(Descriptive Statistics) for numeric variables.**
# Observations

- Statistics were calculated for 79 numeric columns. Text (object) columns such as IP, label, and Timestamp were ignored.

- All columns have a count of 250007. There are no missing values ​​in the numeric columns.

- Some columns, such as Total Length of Fwd Packet and Fwd Packet Length Mean, have minimum values ​​of 0.0, indicating:

- The presence of very short flows or no data (zero-length flows).

- High variability in values ​​(std is very high compared to the mean)—meaning that the data is not uniformly distributed, which is expected in network data with diverse attack patterns.

- The presence of very high maximum values ​​(max), such as:

- Flow Duration up to ~1.19e+09

- Total Fwd Packets up to 704,948
      These are outliers that should be addressed before training.

- The Packet Length columns range from 0 to ~5000 bytes, which makes sense for TCP/UDP streams.




# Step 2: Data Preperation
- Dropping irrelevant columns and rows
- Identifying duplicated columns
- Renaming Columns
- Feature Creation

In [ ]:
df.columns

In [ ]:
drop_cols = [
    "Flow ID", "Src IP", "Dst IP", "Timestamp", "src_path", "cnt", "NeedManualLabel",
    "Fwd URG Flags", "Bwd URG Flags", "CWR Flag Count", "ECE Flag Count",
    "Packet Length Mean",
    "Flow Bytes/s", "Flow Packets/s"
]


# Observations

***This step aims to:***

- Remove irrelevant columns.

- Detect duplicated columns.

 - Rename columns if necessary.

- Create new features later.

- From the drop_cols list, we note that the following columns will be deleted because they are:

                Identifiers that are not useful to the model:
              'Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'src_path', 'cnt', 'NeedManualLabel'

 - Derived or redundant values:
             'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean'

- Low-use or highly correlated flags:
              'Fwd URG Flags', 'Bwd Flags', 'CWR Flag Count', 'ECE Flag Count'

Deleting these columns reduces noise and improves model performance, especially since some of them are merely symbolic or do not add new behavioral information.


drop_cols =

 [
   ##Identifiers / Metadata#

    "Flow ID", "Src IP", "Dst IP", "Timestamp", "src_path", "cnt", "NeedManualLabel",

    # Rare or semi-permanent flags
    "Fwd URG Flags", "Bwd URG Flags", "CWR Flag Count", "ECE Flag Count",

   ##Redundant columns#

    "Packet Length Mean",   # Because it has almost the same meaning as Average Packet Size.

    "Flow Bytes/s",         # It has a lot of info
    "Flow Packets/s"        # It has a lot of info
]


In [ ]:
# Delete unimportant columns
df = df.drop(columns=drop_cols, errors="ignore")

print(" Columns after deletion:", df.shape[1])
print(" Saved columns:", df.columns.tolist()[:20], "...")

In [ ]:
df.shape

***After executing the deletion of non-essential columns, the total number of remaining columns is 73.***
# Observations

The final number of columns is 73, which matches the printed output:

Columns after deletion: 73


The Saved columns list confirms that all critical IDS-related features are still present. These include:
Src Port, Dst Port, Protocol, Flow Duration,
Total Fwd Packet, Total Bwd Packets,
Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min,
and other essential network flow characteristics.

The preservation of these key features indicates that the deletion process did not remove any important analytical or modeling features required for:

Exploratory Data Analysis (EDA)

Machine learning model training

Evasion attack generation

IDS performance evaluation

The number of rows remained unchanged, which confirms that the cleaning step affected only the columns, and no samples were removed.

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.loc[df.duplicated()]

***Finding Missing Values ​​(NaN).***

***Detecting Duplicate Rows.***

# Observations
# **1. Missing Values**

Result from:

      df.isna().sum()

- Shows that all columns have 0 missing values.
- This means that the data is completely complete and contains no NaN or blanks in 250,007 rows and 73 columns.

This is an excellent sign of the quality of the preprocessing and merging process.

- The label column is also null-free, indicating that each flow is correctly sorted (Benign / Attack).

# **2. Duplicate Rows**

Using:

       df.loc[df.duplicated()]

- Shows that there are a number of duplicate rows, all of which share the same values ​​in almost all columns.

- These rows often represent duplicate flows or duplicate records collected from files.

- All numeric values ​​are near or zero, and empty or very short flows (zero-flow) add no value to the model.

In [ ]:
df.loc[df.duplicated(subset=['Src Port'])]


***A custom check for duplicate values ​​is performed based on only one column (Src Port) rather than all columns.***

# Observations

- Any two rows with the same Src Port will be considered "duplicates" even if the other values ​​are different.

- The result shows a large number of rows (approximately 187,561 duplicate rows).

- This is a large number, but it's not necessarily an error, because:

- The same Src Port can be used in different connections (multiple TCP sessions).

- Therefore, these results do not indicate true duplication of data, but rather the reuse of the same port.

- Many rows contain zero values ​​(flow lengths and packet sizes = 0), which could indicate:

- Incomplete flows or very short connection attempts (e.g., handshake only).

- The Protocol column is almost fixed at 6 (i.e., TCP), indicating that most of these connections are over TCP, not UDP.

In [ ]:
# cheking an example duplicate
df.query('`Src Port` == 49153')


***Manually checking for duplicate ports (Src Port == 49153) to determine whether the values ​​are actually duplicates or logically different.***

# Observations

- Multiple rows with the same port (49153) are present.

- However, the values ​​in other columns (e.g., Active Mean, Idle Mean, label) are different, so they are not actually duplicate rows.

- Variation of values ​​in the label between:

"DoS SYN Flood"

"ICMP Fragmentation"

- indicates that the same port was used in different types of attacks or scenarios—a common occurrence in network environments.

- Values ​​for Average Packet Size, Fwd Segment Size Avg, etc., vary slightly between rows, reinforcing the idea that these are different sessions that simply share the same port number.

- A recurring pattern in Fwd/Bwd Bytes = 0 may indicate incomplete or rapidly terminated flows (common in DoS data).

So the result is here:
Replication in Src Port does not mean replication in flows, but rather reusing a port in different sessions.

In [ ]:
df = df.loc[~df.duplicated(subset=['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
       'Total Bwd packets', 'Total Length of Fwd Packet',
       'Total Length of Bwd Packet', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow IAT Mean', 'Flow IAT Std',
       'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean',
       'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total',
       'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
       'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count',
       'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',
       'Down/Up Ratio', 'Average Packet Size', 'Fwd Segment Size Avg',
       'Bwd Segment Size Avg', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg',
       'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg', 'Bwd Packet/Bulk Avg',
       'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Fwd Bytes',
       'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'FWD Init Win Bytes',
       'Bwd Init Win Bytes', 'Fwd Act Data Pkts', 'Fwd Seg Size Min',
       'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean',
       'Idle Std', 'Idle Max', 'Idle Min'])] \
    .reset_index(drop=True).copy()

In [ ]:
df.shape

***Duplicate checking using a large set of columns to identify rows that are actually identical across all important properties in the flow.***

# Observations

- Only retain non-duplicate rows (~df.duplicated(...)).

- Reset the index (reset_index) to avoid truncated row numbers after the deletion.

- After the operation, the result is:

- df.shape → (250001, 73)

- That is, only 6 rows were removed out of 250,007, which means that:

  The data was originally very clean.

- The number of actual duplicates was very small compared to the total size (≈ 0.0024%).

- The unchanged number of columns confirms that the deletion was limited to rows only, without any structural modification.

# Step 3: Feature Understanding
(Univariate analysis)

- Plotting Feature Distributions
    - Histogram
    - KDE
    - Boxplot

In [ ]:
df['Protocol'].value_counts()

# Observations
- 6 → TCP (Transmission Control Protocol)
This is the protocol used in most attacks, such as DoS, DDoS, and Brute Force.

- 17 → UDP (User Datagram Protocol)
This is used in some attacks, such as UDP Flood or DNS Amplification.

- 0 → Unknown Protocol
This is a very rare value and may represent:
Corrupted or unidentified data during flow collection.
Or flows with missing header information.
These can be considered outliers or discarded later.

In [ ]:
df['label'].value_counts()

***Class Distribution Analysis***

# Observations

- The dataset is highly imbalanced.

- Most of the samples fall into the DoS and DDoS categories, while the remaining attacks are very rare.

- The Benign category is very small compared to the attacks, which can cause a bias in the model later (class imbalance problem).

- This type of distribution is very common in IoT IDS data because frequent attacks such as DoS/DDoS generate a large number of flows compared to complex attacks (SQLi, XSS).

In [ ]:
df['label'].value_counts() \
 .plot(kind='bar' , title='Label Distribution')
plt.show()

***Visual Analysis of the Label Distribution Plot***

# Observations

     - The DoS SYN Flood category clearly dominates the data.

     - The DoS UDP Flood category follows, then DDoS, while the remaining categories are almost flat (very small).

    - The Benign category is almost invisible compared to the attacks.

    - Some categories (such as SQL Injection, XSS, Web-Based) are so small that they are difficult to detect in the graph.

    - This figure confirms that the data is highly biased toward DoS/DDoS categories.

 - **Analytically:**

If the model were trained directly on this data, it would learn to recognize only the DoS category and ignore the smaller categories.

Therefore, it is necessary to address this flaw later (re-sampling or weighting).

In [ ]:
df['Protocol'].plot(kind='hist' , bins=20, title='Protocol Distribution')
plt.show()

***A histogram showing the distribution of the Protocol column values***

# Observations

- The graph shows the distribution of the number of flows by protocol number.
The corresponding numbers for the protocols according to the IP standard are:

                - 6 → TCP

                - 17 → UDP

                - 0 → Unknown

-*** From the graph, we can see that:***

- The larger column around the value 6 → indicates that most flows rely on the TCP protocol.

- The smaller column around 17 → represents UDP flows.

- The very small column near 0 → indicates that there are a very limited number of flows using unknown or unused protocols.

- The distribution confirms what was previously observed in value_counts() — that TCP represents the largest percentage of network activity in the dataset.

In [ ]:
df['Src Port'].plot(kind='hist', bins=20, title='Src Port Distribution')
plt.show()

# Observations

- Non-uniform distribution:

- There are two clear peaks in frequency:

            The first is at low ports (0–10,000), typically used for well-known services such as HTTP/HTTPS/FTP/SSH.

            The second is at ports between 45,000–55,000, representing ephemeral ports created by devices during outgoing connections.

This means that the data contains a mix of service traffic and random attack connections.

- There are no very clear peaks at specific ports (such as 80 or 443), which may indicate that most of the data is from inbound flows rather than just web servers.

- The overall pattern is distributed across the entire range 0–65535, indicating:

High port diversity.

- The data includes multiple activities (services + attacks).

In [ ]:
df['Src Port'].plot(kind='kde', title='Src Port Distribution')
plt.show()

***Density curve (KDE plot) for the distribution of source ports (Src Port)***

# Observations

- The Kernel Density Estimation (KDE) function plots a smooth curve showing the density distribution of values ​​in a column.

- The shape of the curve has two distinct peaks (bimodal distribution):

- The first peak, in the range 0–10,000, represents well-known ports.

- The second peak, in the range 45,000–55,000, represents dynamic ports.

- This bimodal pattern is very common in network data because it reflects the interaction of client devices with service servers.

- Overall result:
The distribution of source ports is multi-peaked, reflecting a mix of legitimate and attack traffic, a good indicator of the diversity of activity in the dataset.

In [ ]:
df.plot(kind='scatter', x='Src Port', y='Dst Port', title='Src Port vs Dst Port')
plt.show()

**`*Understanding the behavior of connections within the network (especially to identify attack patterns).*`**

# Observations

- The distribution of points shows two clear patterns:

      A wide horizontal axis at low destination ports (e.g., 80, 443, 53, etc.)
     Indicates that many different source ports connect to the same fixed destination port—this is normal behavior for user traffic (client → server).

         A dense area at the top and right (40k–65k)
        Represents connections between ephemeral ports (ephemeral → ephemeral) and is often associated with attacks or automated connections (e.g., DoS or Mirai botnet).

- The presence of clear horizontal or vertical lines:

Solid horizontal lines: Indicate that the same Dst port is frequently targeted (e.g., 80, 443, 8080).

Solid vertical lines: Indicate that the same Src port is used for multiple connections (possibly repeated attacks from the same port).

A very high density in the graph indicates that the data contains a large number of flows.

In [ ]:
df.plot(kind='scatter', x='Idle Mean', y='Idle Std', title='Idle Mean vs Idle Std')
plt.show()

***The graph shows the relationship between Idle Mean and Idle Std (mean and skewed idle periods between flows).***

***These metrics are important because they reflect the regularity or stagnation of connections in the network—and often help distinguish between normal and attack traffic***.

#  Observations

**From the graph, we can see**:

- There is a clear triangular pattern starting at (0,0) and moving up and right.

       The dense portion at the bottom (close to zero) represents short or attack flows (DoS/DDoS) because they have almost no idle periods.

       The higher points represent flows with long and irregular idle periods, which are normal or complex flows (such as Benign or Web-based sessions).

**The relationship is near-linear (positive correlation)**:

          The higher the idle time (Idle Mean), the higher the dispersion (Idle Std).

          This makes sense because long, irregular connections have more variable idle times.

**The presence of some points on the lower line (Idle Std = 0) means**:

          Flows with no idle time variation → Very short connections or constant-paced attack flows (such as a SYN Flood).

- **This graph is very useful for distinguishing**:

Fast attacks (Low Idle Mean + Std)

Normal or complex flows (High Idle Mean + Std)

In [ ]:
df.plot(kind='scatter', x='Idle Max', y='Idle Min', title='Idle Max vs Idle Min')
plt.show()

***The Idle Max vs. Idle Min relationship reveals the nature of the distribution of maximum and minimum idle periods in each flow.***

# Observations

     Idle Max: The longest idle period between packets in a flow.

     Idle Min: The shortest idle period between packets in the same flow.

- Graphic Shape:

         - It shows a clear triangular pattern starting at (0,0) and rising to the right.

         - There is a straight diagonal line (from lower left to upper right) representing the cases where:
                      Idle Max ≈ Idle Min
          i.e., the idle periods in these flows are approximately constant (perfect time uniformity).

- Most points below the line,
Idle Min < Idle Max
make sense because the shortest idle period is always less than the longest period.

- Network Behavior Analysis:

- Points close to the diagonal line: Stable flows (usually normal traffic or long, stable connections).

- Points scattered far from the line: Unstable flows or attacks (DoS or DDoS) with highly variable idle periods.

- The density at the bottom indicates many flows with very short idle periods (possibly frequent flash attacks).

- The relationship between the two variables is clearly positive—the shorter the idle period, the longer the idle period, indicating relative consistency in idle behavior.

In [ ]:
sns.scatterplot(x='Src Port',
                y='Protocol',
                hue= 'label',
                data=df)

plt.legend(loc='upper right')
plt.show()
plt.show()

***A scatterplot linking the source port (Src Port) to the protocol, with the points colored by attack type (label), showing how attacks are distributed across ports and protocols.***

#  Observations

- Observations from the graph:

       -The top line (≈17) represents the UDP protocol and associated attacks, such as:

                  - DoS UDP Flood

                      -  DDoS

            - ICMP Flood (often via UDP or ICMP raw socket)

- The middle line (≈6) represents the TCP protocol and includes the most common types of attacks, such as:

                  - DoS SYN Flood

                   - HTTP Flood

                  - SQL Injection

                   - Brute Force

                    -  Web-Based

                     -   Benign

- The bottom line (≈0) contains very few points—rare or anomalous, possibly caused by unclassified flows or logging errors.

- Src Port Distribution:

              The ports are roughly distributed between (0–65535), but most are within the usual ranges of (0–10k) and (45k–65k).

              There is little overlap between protocols, reflecting that each attack often relies on a single underlying protocol.

- TCP attacks such as SYN/HTTP Flood appear on line 6, the most frequently occurring.

- UDP-based attacks appear clearly and separately on line 17.

- The Benign category is distributed across both protocols, but with less frequency.

In [ ]:
sns.pairplot(df,
             vars=['Src Port'   ,'Dst Port'     ,
                       'Protocol'       ,'Flow Duration'],
                hue='label')
plt.show()

***A pairplot shows the relationships between a set of variables at a time, with the points colored by attack type (label).***

# Observations

- General Analysis of the Graph:

The overall shape shows the distribution of each category (attack type) across ports, protocols, and time.

# ***Src Port vs Dst Port***

The points overlap, but we see two clear ranges:

Random source ports (40k–60k) vs. fixed destination ports (80, 443, etc.).

Some categories, such as DoS SYN Flood and UDP Flood, are highly concentrated in this range, confirming their attack pattern.

#  ***Protocol vs Ports***

The horizontal lines at 6 and 17 reflect that the attacks are clearly distributed across TCP and UDP.

The categories vary by protocol:

TCP → SYN Flood, HTTP Flood, Web-Based, SQL Injection

UDP → UDP Flood, DDoS, ICMP Flood

#  ***Flow Duration***

Most flows are very short (at low values ​​on the horizontal axis).

The Benign and Web-Based categories span longer durations (normal connections are typically longer and more stable).

The DoS/DDoS attack categories are concentrated at very low Duration values ​​because they are frequent and rapid flows.

In [ ]:
df_corr = df[['Src Port','Dst Port','Protocol','Flow Duration']].dropna().corr()
df_corr

In [ ]:
df[['Total Fwd Packet', 'Total Bwd packets'   ,'Total Length of Fwd Packet',  'Total Length of Bwd Packet' ]].dropna().corr()

***Correlation Matrix between two sets of properties—one for ports, protocol, and time, and the other for packets and their length.***

# Observations

- Correlation between Src Port and Dst Port = -0.18

          Weak and negative correlation (mild inverse)
          Natural because the source and destination ports are often different between each connection.

- Flow Duration and Dst Port = -0.307

              Medium and negative correlation:
              Means that flows going to certain ports tend to be shorter (e.g., DoS attacks on ports 80/443).
              While long connections tend to go to other or variable ports.

- Protocol and Flow Duration = 0.175

             Weak and positive correlation:
             The protocol may have little effect on flow duration, e.g., UDP is typically shorter than TCP.


- In general: Correlations are weak, meaning that each variable carries different information — this is very useful for the model because it reduces information redundancy.

In [ ]:
sns.heatmap(df_corr, annot=True)

Correlation Heatmap between four key features:
Src Port, Dst Port, Protocol, and Flow Duration.

# Observations

          The heatmap illustrates the strength and direction of the relationship:

                Light color ≈ Strong positive correlation (values ​​close to +1)

                Dark color ≈ Negative or weak correlation (values ​​close to -1 or 0)

- The figure confirms that ports, protocol, and time are semi-independent variables.

- This is excellent for building a machine learning model because:

              It reduces the risk of multicollinearity (repeating the same information).

              It ensures that each feature adds new information to the model.

It is notable that Flow Duration is the most distinct variable, as it is correlated (positively or negatively) with more than one variable,
making it a sensitive indicator of the type of connection (Benign or Attack).

# Step 5: Ask a Question about the data
"What are the attack types with the highest average Src Port values (minimum of 10 flows)?"


In [ ]:
ax = df.query('label != "Benign"') \
       .groupby('label')['Src Port'] \
       .agg(['mean', 'count']) \
       .query('count >= 10') \
       .sort_values('mean', ascending=False)['mean'] \
       .plot(kind='barh', figsize=(12, 5), title='Average Src Port by Attack Type')

ax.set_xlabel('Average Src Port')
plt.show()


# Observations

- The highest source port averages were for the following attacks:

                 DNS Spoofing

                 ICMP Fragmentation

                 DoS HTTP Flood

                 SQL Injection

                 Scan / Mirai

- The lowest were:

                 Web-Based

                 DDoS

                 Brute Force

- High port averages (≈ 40,000) indicate the use of ephemeral ports—typically used in automated or random attacks (such as spoofing and scanning).

- Low port averages (≈ 20,000–25,000) are for attacks that are more targeted toward specific servers (such as Web-Based and DDoS).

- The variation in port averages reflects that each attack type tends to use different ranges of source ports.

- Web-Based and DDoS have the lowest averages because these attacks target specific ports (80/443) and often originate from relatively stable ports.

# ***XGBOOST***

In [ ]:
# Importing Essential Libraries
# Initialization & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# XGBoost Classification Model
from xgboost import XGBClassifier

# Creating an object from the model
XGBClassifier()


# Reading the sample file into DataFrame
# Preview the first 5 rows to quickly see how the data looks
# Column information: type, non-empty values, memory usage...
df2 = pd.read_csv('/content/drive/MyDrive/sample_stratified_250k_ready.csv')
df2.head()
df2.info()


# Observations

- The dataset contains 250,001 network flow records and 73 features in total.  
- Feature types include 42 float64 (continuous), 30 int64 (discrete), and 1 object (categorical: “label”).  
- No missing or null values are present — all features have complete data coverage.  
- The “label” column represents the target class for classification.  
- Features capture both forward (Fwd) and backward (Bwd) traffic directions, reflecting flow-based characteristics.

In [ ]:
# Data Splitting & Label Encoding

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df2.drop('label', axis=1)
y = df2['label']

# Encode the target variable 'y' to numerical labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Divide the data into training and testing at a ratio of 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=0, stratify=y_encoded

)


# Apply SMOTE Oversampling on Training Data
from imblearn.over_sampling import SMOTE


sm = SMOTE(   random_state=42,         # random_state=42 ensures reproducibility (same results every run)
                k_neighbors=5          # k_neighbors=5 defines how many nearest neighbors are used to generate synthetic samples
              )



# Apply SMOTE only on the training data
# This step generates synthetic samples for minority classes
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

# Display the number of samples before and after balancing
from collections import Counter
print("Before:", Counter(y_train))
print("After :", Counter(y_train_sm))

# Observations
- Features/target separated: X = df2.drop('label', axis=1), y = df2['label'].

- Target encoded with LabelEncoder() to numeric classes (multi-class ready).

- Stratified 80/20 split (stratify=y_encoded) keeps the original class proportions in the test set.

- SMOTE applied only on the training set (sm.fit_resample(X_train, y_train)), avoiding leakage into test data.

- Class distribution after SMOTE becomes balanced per class (counts roughly equal across all classes), improves minority-class learning.

- Test set remains untouched (original, imbalanced reality), fair evaluation.

In [ ]:
# (Model Training & Evaluation)
from sklearn.metrics import accuracy_score

# Creating an XGBoost model and defining hyperparameters
model = XGBClassifier(

    learning_rate=0.1,     # Learning rate (how much the model changes each time during training)
    max_depth=6,           # Depth of each tree (controlling model complexity)
    subsample=0.8 ,
    n_estimators=200 ,

    tree_method='hist',  # Changed from 'gpu_hist' to 'hist'
    n_jobs=-1,
    random_state=42
    )

# Training the model on training data (applying SMOTE)
model.fit(X_train_sm, y_train_sm)


# Predicting categories (labels)

# On the test data
y_predict= model.predict(X_test)
# On training data
y_train_predict = model.predict(X_train_sm)

# Accuracy calculation for training and testing
print("Train accuracy", accuracy_score(y_train_sm, y_train_predict))
print("Test accuracy",accuracy_score(y_test,y_predict))

# Observations
- The XGBoost model was trained on SMOTE-balanced data using 200 estimators and a maximum tree depth of 8.  
- Training accuracy reached 0.883, while testing accuracy reached 0.946 — indicating strong generalization and no overfitting.  

- Training accuracy is lower than testing accuracy because SMOTE added new synthetic samples, making training a bit harder. This helps the model avoid overfitting and perform better on test data.

- The learning rate (0.1) provided stable convergence without oscillations or performance drops.  
- Subsampling (0.8) added randomness, reducing the risk of overfitting and improving robustness.  
- The “hist” tree method optimized performance for CPU-based training.  
- Overall, the model demonstrates high detection capability and stability on unseen IoT data after class balancing.

In [ ]:
# Model Evaluation
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# Test predictions
y_pred = model.predict(X_test)

# Extracting the actual class names (labels) from y_test
labels = np.unique(y_test)                      # Numbers that represent categories (0, 1, 2,...)
names = le.inverse_transform(labels)            # Convert them to their original names (Benign, DoS, DDoS...)

# Displaying the performance report for each category (Precision, Recall, F1)
print(classification_report(
    y_test, y_pred,
    labels=labels,
    target_names=names,
    digits=4
))

# Print the overall accuracy on the test set
print("Accuracy:", accuracy_score(y_test, y_pred))

# Creating a Confusion Matrix
# Explains where the model went wrong — how often it confused different categories
cm = confusion_matrix(y_test, y_pred, labels=labels)

# Visualize and display the Confusion Matrix to understand the model’s classification errors
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=names)
disp.plot(xticks_rotation=45, cmap='Blues')
plt.tight_layout()
plt.show()


# Observations
• Evaluation performed using classification report and confusion matrix.  
• Overall accuracy on the test set reached 0.9468, confirming strong generalization.  
• Major attack types (DoS, DDoS, SYN Flood, HTTP Flood) achieved very high precision and recall.  
• Minority attacks such as XSS, SQL Injection, and Brute Force show low detection rates due to limited training samples.  
• Macro F1-score (0.52) is lower than weighted F1 (0.95), reflecting the imbalance across attack categories.  
• The confusion matrix shows clear diagonal dominance for DoS-related classes, indicating reliable detection of high-frequency attacks.  
• Some misclassifications occur among similar attack patterns (e.g., HTTP Flood ↔ SYN Flood ↔ UDP Flood).  
• Overall, the XGBoost model demonstrates high accuracy and stable detection on IoT-IDS data, even with diverse attack types.


# 0) SETUP: Data and Target Model

In [ ]:
import torch
import numpy as np

# Convert balanced training data into tensors
y_train_torch = torch.tensor(y_train_sm, dtype=torch.long)
y_test_torch  = torch.tensor(y_test, dtype=torch.long)


# Convert test data to tensors
X_test_torch = torch.tensor(X_test.values, dtype=torch.float32)
y_test_torch = torch.tensor(y_test, dtype=torch.long)


# SCRIPT 1 — BUILD SURROGATE MODEL (PyTorch MLP)

In [ ]:
import torch.nn as nn
import torch.optim as optim

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train_sm))

learning_rate = 0.005
epochs = 50
batch_size = 128
dropout_rate = 0.5

class SurrogateMLP(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rate):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64)
        )
        self.output_layer = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.output_layer(x)

surrogate = SurrogateMLP(input_dim, num_classes, dropout_rate)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(surrogate.parameters(), lr=learning_rate, weight_decay=1e-5)

# SCRIPT 2 — TRAIN SURROGATE WITH PSEUDO-LABELING

In [ ]:

surrogate.train()
print("\nTraining surrogate model to approximate XGBoost decisions...\n")

for epoch in range(epochs):
    perm = torch.randperm(X_train_torch.size(0))
    total_loss = 0
    num_batches = 0

    for i in range(0, X_train_torch.size(0), batch_size):
        idx = perm[i:i+batch_size]
        batch_x = X_train_torch[idx]

        # Get pseudo-labels from XGBoost
        teacher_pred = model.predict(batch_x.numpy())  # <-- XGBoost model here
        batch_y = torch.tensor(teacher_pred, dtype=torch.long)

        optimizer.zero_grad()
        out = surrogate(batch_x)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss={total_loss/num_batches:.4f}")

print("Surrogate training completed.\n")

*To reduce excessive computation while maintaining the surrogate model’s ability to mimic the XGBoost classifier, the surrogate MLP was trained for 50 epochs, during which the loss remained stable and continued to gradually decrease. As shown in the training log, the loss started at approximately 0.4449 and converged toward 0.4376 by the final epoch. This consistent but slow decline indicates that the surrogate successfully learned an approximate decision boundary without signs of overfitting or instability.
The smooth convergence trend confirms that the surrogate model captured the essential decision patterns of the XGBoost target, providing a reliable white-box approximation to enable gradient-based attacks such as PGD.*

# 3) PGD ATTACK PARAMETERS


In [ ]:
eps = 0.6                     # Maximum allowed perturbation magnitude
alpha = 0.01                  # Step size for each PGD iteration
T_list = [10, 20, 50, 80, 100]  # Number of PGD iterations to evaluate

#4) PGD ATTACK FUNCTION

In [ ]:
def pgd_attack(x, y, model, eps, alpha, T):
    model.eval()
    x_adv = x.clone().detach()
    for t in range(T):
        x_adv.requires_grad = True
        out = model(x_adv)
        loss = criterion(out, y)
        model.zero_grad()
        loss.backward()
        grad = x_adv.grad.data
        x_adv = x_adv + alpha * grad.sign()
        x_adv = torch.min(torch.max(x_adv, x - eps), x + eps)
        x_adv = x_adv.detach()
    return x_adv

# 5) RUN PGD ATTACK ON XGBOOST

In [ ]:
from sklearn.metrics import accuracy_score

results = []
print("Running PGD attacks on the black-box XGBoost model...\n")

for T in T_list:
    print(f"Performing PGD with {T} iterations...")
    X_adv = pgd_attack(X_test_torch, y_test_torch, surrogate, eps, alpha, T)
    y_pred_adv = model.predict(X_adv.numpy())  # <-- XGBoost evaluation
    acc_adv = accuracy_score(y_test, y_pred_adv)
    results.append((T, acc_adv))
    print(f"  XGBoost Accuracy on Adversarial Samples: {acc_adv:.4f}\n")

print("PGD attack evaluations completed.\n")
print("=========== FINAL RESULTS ===========")
for T, acc in results:
    print(f"T = {T} → XGBoost Accuracy = {acc:.4f}")

*To evaluate the robustness of the black-box XGBoost model against adversarial perturbations, we performed PGD attacks with different iteration counts. As the number of PGD steps increased, the model’s accuracy consistently decreased, indicating a higher vulnerability to stronger iterative attacks. For low-intensity attacks (10–20 steps), the model experienced only a mild reduction in accuracy, suggesting partial resilience to small perturbations. However, as the number of steps increased to 50, 80, and 100, the degradation became more pronounced, revealing that PGD is able to progressively craft more harmful adversarial samples. Notably, the accuracy plateaued between 80 and 100 steps, indicating attack saturation, where additional iterations no longer yield significant performance drops. Overall, these results demonstrate that XGBoost is moderately robust to weak PGD attacks but substantially vulnerable to stronger ones, emphasizing the need for enhanced adversarial defense mechanisms in IDS environments.*

### Enhanced evaluation + automatic saving utilities

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from textwrap import wrap


def _ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
    return path


def save_confusion_matrix_figure(cm, classes, outpath, title="Confusion Matrix", cmap=None):
    fig, ax = plt.subplots(figsize=(6, 5))
    if cmap is None:
        cmap = "Blues"
    sns.heatmap(cm, annot=True, fmt=".2f", ax=ax, cmap=cmap, cbar=True)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def evaluate_evasion_performance(
    y_true,
    y_pred_clean,
    y_pred_adv,
    y_score_clean=None,
    y_score_adv=None,
    normal_class_label=0,
    model_name="Model",
    attack_name="Attack",
    save=True,
    save_dir="evaluation_results",
    prefix=None,
    classes=None
):
    """
    Enhanced unified evaluation for IDS under evasion attacks, with optional automatic saving.
    Returns:
      df_metrics (pd.DataFrame) rounded to 4 decimals,
      save_info (dict) with paths to saved files if save=True (otherwise None)
    """
    sns.set_theme(style="whitegrid", context="notebook")
    plt.rcParams["figure.dpi"] = 140

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if prefix is None:
        folder_name = f"{timestamp}__{model_name}__{attack_name}".replace(" ", "_").replace("/", "-")
    else:
        folder_name = f"{timestamp}__{prefix}".replace(" ", "_").replace("/", "-")

    out_base = os.path.join(save_dir, folder_name)
    if save:
        _ensure_dir(out_base)

    print(f"\n{'='*12} {model_name} — {attack_name} Evaluation {'='*12}\n")

    # ===== Metadata Overview =====
    print("📋 Metadata Overview:")
    print(f" - Timestamp: {timestamp}")
    print(f" - Model: {model_name}")
    print(f" - Attack: {attack_name}")
    print(f" - Total Samples: {len(y_true)}")
    print(f" - Save Directory: {out_base if save else 'Saving Disabled'}")
    print("="*55 + "\n")

    # ===== 1. Compute metrics =====
    metrics = {}
    for name, y_pred in {"Clean": y_pred_clean, "Adversarial": y_pred_adv}.items():
        metrics[name] = {
            "Accuracy": float(accuracy_score(y_true, y_pred)),
            "Precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
            "Recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
            "F1-score": float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
        }

    asr = float(np.mean(np.array(y_pred_clean) != np.array(y_pred_adv)))
    metrics["Adversarial"]["ASR"] = asr

    # Optional AUC (for Normal class)
    if y_score_clean is not None and y_score_adv is not None:
        try:
            metrics["Clean"]["AUC"] = float(
                roc_auc_score(
                    (np.array(y_true) == normal_class_label).astype(int),
                    np.array(y_score_clean)[:, normal_class_label]
                )
            )
            metrics["Adversarial"]["AUC"] = float(
                roc_auc_score(
                    (np.array(y_true) == normal_class_label).astype(int),
                    np.array(y_score_adv)[:, normal_class_label]
                )
            )
        except Exception as e:
            print("AUC calculation skipped:", e)

    # ===== 2. Tabular summary =====
    df_metrics = pd.DataFrame(metrics).T
    display(df_metrics.round(4))

    # ===== 3. Classification report (Adversarial only) =====
    clf_report_dict = classification_report(y_true, y_pred_adv, output_dict=True, zero_division=0)
    clf_report_str = classification_report(y_true, y_pred_adv, digits=4, zero_division=0)
    print("\nDetailed Classification Report (Adversarial Predictions):\n")
    print(clf_report_str)

    # ===== 4. Confusion matrices =====
    cm_clean = confusion_matrix(y_true, y_pred_clean, normalize="true")
    cm_adv = confusion_matrix(y_true, y_pred_adv, normalize="true")

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(cm_clean, cmap="Blues", ax=ax[0], annot=False)
    ax[0].set_title("Normalized Confusion Matrix — Clean")
    ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")

    sns.heatmap(cm_adv, cmap="Reds", ax=ax[1], annot=False)
    ax[1].set_title("Normalized Confusion Matrix — Adversarial")
    ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")

    plt.suptitle(f"{model_name} — {attack_name}: Class Confusion Comparison", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # ===== 5. Metric comparison plots =====
    df_compare = df_metrics[["Accuracy", "Precision", "Recall", "F1-score"]].T
    df_compare["Δ (drop)"] = df_compare["Clean"] - df_compare["Adversarial"]

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    df_compare[["Clean", "Adversarial"]].plot(kind="bar", ax=ax[0])
    ax[0].set_title("Metric Comparison Before vs After Attack")
    ax[0].set_ylabel("Score")
    ax[0].legend(loc="lower right")

    sns.barplot(x=df_compare.index, y=df_compare["Δ (drop)"], ax=ax[1])
    ax[1].set_title("Performance Drop per Metric")
    ax[1].set_ylabel("Δ = Clean - Adversarial")
    ax[1].axhline(0, color="gray", linestyle="--")

    plt.suptitle(f"{model_name} — {attack_name}: Performance Impact Overview", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

    # ===== 6. ASR visualization =====
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(["Attack Success Rate"], [asr * 100])
    ax.set_title("Evasion Success (ASR)")
    ax.set_ylim(0, 100)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

    # ===== 7. Text summary =====
    print("\nSummary Insights:")
    print(f" - Clean Accuracy: {metrics['Clean']['Accuracy']:.4f}")
    print(f" - Adversarial Accuracy: {metrics['Adversarial']['Accuracy']:.4f}")
    print(f" - Attack Success Rate (ASR): {asr*100:.2f}%")
    print(f" - Drop in Accuracy: {(metrics['Clean']['Accuracy']-metrics['Adversarial']['Accuracy'])*100:.2f}%")
    print(f" - Drop in F1-score: {(metrics['Clean']['F1-score']-metrics['Adversarial']['F1-score'])*100:.2f}%")
    if "AUC" in metrics["Clean"]:
        delta_auc = metrics["Clean"]["AUC"] - metrics["Adversarial"]["AUC"]
        print(f" - AUC Change (Normal class): {delta_auc:+.4f}")
    print(f"\nEvaluation completed for {model_name} under {attack_name}.\n")

    # ===== 8. Saving outputs =====
    save_info = None
    if save:
        save_info = {}
        metrics_excel = os.path.join(out_base, "metrics_summary.xlsx")
        metrics_csv = os.path.join(out_base, "metrics_summary.csv")
        df_metrics.round(6).to_excel(metrics_excel)
        df_metrics.round(6).to_csv(metrics_csv, index=True)
        save_info["metrics_excel"] = metrics_excel
        save_info["metrics_csv"] = metrics_csv

        clf_json = os.path.join(out_base, "classification_report_adversarial.json")
        clf_txt = os.path.join(out_base, "classification_report_adversarial.txt")
        with open(clf_json, "w", encoding="utf-8") as f:
            json.dump(clf_report_dict, f, indent=2)
        with open(clf_txt, "w", encoding="utf-8") as f:
            f.write(clf_report_str)
        save_info["classification_report_json"] = clf_json
        save_info["classification_report_txt"] = clf_txt

        cm_clean_npy = os.path.join(out_base, "cm_clean.npy")
        cm_adv_npy = os.path.join(out_base, "cm_adv.npy")
        np.save(cm_clean_npy, cm_clean)
        np.save(cm_adv_npy, cm_adv)
        save_info["cm_clean_npy"] = cm_clean_npy
        save_info["cm_adv_npy"] = cm_adv_npy

        classes_arg = classes if classes is not None else [str(i) for i in range(cm_clean.shape[0])]
        cm_clean_png = os.path.join(out_base, "cm_clean.png")
        cm_adv_png = os.path.join(out_base, "cm_adv.png")
        save_confusion_matrix_figure(cm_clean, classes_arg, cm_clean_png, title="Normalized Confusion Matrix - Clean", cmap="Blues")
        save_confusion_matrix_figure(cm_adv, classes_arg, cm_adv_png, title="Normalized Confusion Matrix - Adversarial", cmap="Reds")
        save_info["cm_clean_png"] = cm_clean_png
        save_info["cm_adv_png"] = cm_adv_png

        preds_npz = os.path.join(out_base, "predictions_and_labels.npz")
        np.savez_compressed(preds_npz,
                            y_true=np.array(y_true),
                            y_pred_clean=np.array(y_pred_clean),
                            y_pred_adv=np.array(y_pred_adv))
        save_info["preds_npz"] = preds_npz

        meta = {
            "model_name": model_name,
            "attack_name": attack_name,
            "timestamp": timestamp,
            "rows": int(len(y_true)),
            "asr": asr,
            "paths": save_info
        }
        meta_path = os.path.join(out_base, "metadata.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)
        save_info["metadata"] = meta_path

        print(f"Saved evaluation outputs to: {out_base}")

    return df_metrics.round(4), save_info

In [ ]:
df_eval, save_info = evaluate_evasion_performance(
   y_true=y_test,                                       # true labels
   y_pred_clean = model.predict(X_test),                                                   # clean predictions
    y_pred_adv=y_pred_adv,                               # adversarial predictions from PGD
    y_score_clean=None,                                  # RF has no probabilities here
    y_score_adv=None,                                    # same as above
    normal_class_label=0,                                # Benign label (0)
    model_name="XGBOOST_Default_SMOTE",             # model identifier
    attack_name="PGD_eps0.6_alpha0.01_steps100",         # describe attack settings
    save=True,                                           # save outputs on disk
    save_dir="eval_outputs_PGD",                         # output directory
    prefix="PGD_xgb_Eval",                                # folder prefix
    classes=[str(c) for c in np.unique(y_test)]          # class names for plotting
)

# --- Output summary ---
print("\n✅ Evaluation completed successfully!")
print("📊 Metrics DataFrame:")
display(df_eval)

print("\n📁 Saved files summary:")
for k, v in save_info.items():
    print(f" - {k}: {v}")